In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ======================
# CONFIG
# ======================
TOP_N = 200   # 🔥 tăng recall
TOP_K = 5
DB_CONFIG = {
    "host": "aws-1-ap-northeast-2.pooler.supabase.com",  # 🔥 đổi lại
    "port": 6543,                          # 🔥 không dùng 6543 nữa
    "database": "postgres",
    "user": "postgres.ypkwqsbsunlvpoqdadbq",
    "password": "5$eAK8EV4S+gsKj",
    "sslmode": "require"
}
# ======================
# USER CONTEXT
# ======================
user_context = {
    "user_id": 0,
    "lat": 10.75,
    "lon": 106.67,
    "budget_level": 2
}

# ======================
# PARSED INTENT + SLOT
# ======================
parsed = {
    "intent": "find_restaurant",
    "slots": {
        "food": "thịt bò",
        "price": 2,
        "distance_km": 5
    }
}
food = parsed["slots"].get("food")
# ======================
# CONNECT DB
# ======================
conn = psycopg2.connect(**DB_CONFIG)

# ======================
# STEP 1: BUILD SQL DYNAMIC
# ======================
def build_sql(parsed, user_context):

    user_lat = user_context["lat"]
    user_lon = user_context["lon"]

    distance_km = parsed["slots"].get("distance_km", 5)
    distance_threshold = distance_km / 111  # convert km → degree

    price = parsed["slots"].get("price")

    # 🔥 optional price
    if price is not None:
        price_condition = f"ABS(r.price_level - {price}) <= 1"
    else:
        price_condition = "TRUE"

    sql = f"""
    SELECT 
        r.id,
        r.name,
        r.price_level,
        r.latitude,
        r.longitude,

        SQRT(POWER(r.latitude - {user_lat}, 2) + 
             POWER(r.longitude - {user_lon}, 2)) AS distance,

        re.content,
        re.embedding,

        COALESCE(AVG(ur.rating), 0) as rating

    FROM restaurants r
    JOIN restaurant_embeddings re ON r.id = re.restaurant_id
    LEFT JOIN user_ratings ur ON r.id = ur.restaurant_id

    GROUP BY r.id, re.content, re.embedding

    HAVING 
        SQRT(POWER(r.latitude - {user_lat}, 2) + 
             POWER(r.longitude - {user_lon}, 2)) < {distance_threshold}
        AND {price_condition}

    LIMIT {TOP_N}
    """
    
    return sql

# ======================
# STEP 2: LOAD DATA
# ======================
sql_query = build_sql(parsed, user_context)
df = pd.read_sql(sql_query, conn)

df["embedding"] = df["embedding"].apply(lambda x: np.array(json.loads(x)))

print(f"✅ SQL returned {len(df)} candidates")

# ======================
# STEP 3: OPTIONAL KEYWORD FILTER (slot food)
# ======================


original_df = df.copy()



if food:
    filtered_df = df[df["content"].str.contains(food, case=False, na=False)]

    if len(filtered_df) > 0:
        print(f"✅ After food filter: {len(filtered_df)}")
        df = filtered_df
    else:
        print("⚠️ Không tìm thấy món → fallback semantic")
        df = original_df



# ======================
# STEP 4: TF-IDF
# ======================
if len(df) == 0:
    print("❌ No candidates → skip TF-IDF")
    tfidf_result = pd.DataFrame()
else:
    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(df["content"])

def search_tfidf(query):
    query_vec = tfidf.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    tmp = df.copy()
    tmp["score"] = scores

    return tmp.sort_values("score", ascending=False).head(TOP_K)

# ======================
# STEP 5: EMBEDDING
# ======================
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def search_embedding(query):
    query_emb = model.encode(query)

    scores = [
        cosine_similarity([query_emb], [doc_emb])[0][0]
        for doc_emb in df["embedding"]
    ]

    tmp = df.copy()
    tmp["score"] = scores

    return tmp.sort_values("score", ascending=False).head(TOP_K)

# ======================
# TEST
# ======================
query = "Tìm quán thịt bò, giá tầm trung, gần đây cho tôi"

tfidf_result = search_tfidf(query)
embedding_result = search_embedding(query)

# ======================
# PRINT
# ======================
print("\n===== TF-IDF RESULT =====")
print(tfidf_result[["name", "rating", "score"]])

print("\n===== EMBEDDING RESULT =====")
print(embedding_result[["name", "rating", "score"]])


C:\Users\Admin\AppData\Local\Temp\ipykernel_15892\86926351.py:134: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql_query, conn)


✅ SQL returned 200 candidates
✅ After food filter: 32

===== TF-IDF RESULT =====
                               name    rating     score
25                     Quán Phở xào  2.696839  0.078833
3   Quán Mì nưa xào kim chi thịt bò  2.825000  0.059245
91   Quán Thịt bò xào kiểu Hàn Quốc  2.618269  0.058243
60  Quán Bún thịt heo xiên sả nướng  2.736494  0.057235
98           Quán Pancake dành dành  2.623512  0.055501

===== EMBEDDING RESULT =====
                                name    rating     score
76               Quán Dê nướng sa tế  2.703571  0.508371
3    Quán Mì nưa xào kim chi thịt bò  2.825000  0.504227
173         Quán Gỏi Thịt Bò Cà Pháo  3.079301  0.491572
60   Quán Bún thịt heo xiên sả nướng  2.736494  0.489517
23       Quán Canh thịt bò cải xoong  2.685484  0.479503


In [45]:
# 1) Xem tất cả tên cột
print(embedding_result.columns.tolist())
# print(embedding_result[["id","name", "rating", "score","content"]])
restaurant_ids = embedding_result["id"].tolist()
def get_review_counts_batch(conn, restaurant_ids):
    query = f"""
    SELECT restaurant_id, COUNT(*) as review_count
    FROM user_ratings
    WHERE restaurant_id IN %s
    GROUP BY restaurant_id
    """

    cur = conn.cursor()
    cur.execute(query, (tuple(restaurant_ids),))
    rows = cur.fetchall()
    cur.close()

    # convert → dict
    review_map = {rid: cnt for rid, cnt in rows}

    return review_map
review_map = get_review_counts_batch(conn, restaurant_ids)
print(review_map)



['id', 'name', 'price_level', 'latitude', 'longitude', 'distance', 'content', 'embedding', 'rating', 'score']
{736: 28, 616: 32, 873: 31, 651: 31, 709: 29}


In [46]:
from collections import defaultdict
def clean_and_merge_tags(tags):
    return list(set(
        t.strip().lower()
        for t in tags
        if t and isinstance(t, str)
    ))

def get_tags_batch(conn, restaurant_ids):
    query = """
    SELECT restaurant_id, tags
    FROM menus
    WHERE restaurant_id = ANY(%s)
    """

    cur = conn.cursor()
    cur.execute(query, (restaurant_ids,))
    rows = cur.fetchall()
    cur.close()

    tag_map = defaultdict(list)

    for rid, tags in rows:
        if tags:
            tag_map[rid].extend(tags)

    # clean luôn
    for rid in tag_map:
        tag_map[rid] = clean_and_merge_tags(tag_map[rid])

    return tag_map

tag_map = get_tags_batch(conn, restaurant_ids)
print(tag_map)
def get_open_time_slot_batch(conn, restaurant_ids):
    query = """
    SELECT id, open_time_slot
    FROM restaurants
    WHERE id = ANY(%s)
    """

    cur = conn.cursor()
    cur.execute(query, (restaurant_ids,))
    rows = cur.fetchall()
    cur.close()

    # convert → dict
    time_map = {rid: slot for rid, slot in rows}

    return time_map
time_map = get_open_time_slot_batch(conn, restaurant_ids)
print(time_map)


defaultdict(<class 'list'>, {616: ['món mì xào', 'món ngon từ hàu', 'mì xào', 'món cháo ngon', 'canh sườn', 'cam', 'nha đam', 'thịt kho', 'món kho', 'thịt bò', 'cháo', 'món cháo ngon dễ nấu', 'mì nưa', 'trứng cút kho', 'hàu', 'cách kho thịt ngon', 'món cháo', 'ba chỉ', 'kim chi', 'chả lụa', 'mì xào kim chi'], 651: ['cá chiên giòn', 'cá cơm', 'canh thịt bò', 'cách kho cá', 'món canh dễ nấu', 'canh cải xoong', 'hải sản', 'món xào', 'cải xoong', 'mực', 'canh ga', 'nước cốt dừa', 'món gà dễ làm', 'gà quay'], 873: ['gỏi thịt bò cà pháo', 'gỏi đu đủ tôm tai heo', 'sữa chua úp ngược', 'ốc hương rang muối sợi', 'thạch sữa bắp lá dứa'], 709: ['bún thịt nướng', 'nghệ thuật nấu ăn ngon gia đình', 'thịt kho', 'thịt bò kho', 'cách kho thịt ngon', 'thit bo kho tau', 'hải sản', 'lẩu chua cay', 'áp chảo', 'món bún', 'món chiên', 'thịt heo nướng sả', 'mực', 'lẩu bò', 'thịt bò'], 736: ['lá cách', 'bò nướng', 'rau cang cua', 'nấm', 'dê nướng', 'cốt lết chiên', 'món dê', 'món xào chay', 'cốt lết', 'goi bo

In [47]:


def build_restaurant(row, tags, review_count, open_time_slot):
    return {
        "id": row["id"],
        "lat": row["latitude"],
        "lng": row["longitude"],
        "price_level": row["price_level"],
        "rating": row["rating"],
        "review_count": review_count,
        "categories": tags,
        "semantic_score": row["score"],
        "open_time_slot": open_time_slot
    }
restaurants = []
for _, row in embedding_result.iterrows():
    print(row["name"], row["score"])
    print("Categories:", tag_map.get(row["id"], []))
    print("Review count:", review_map.get(row["id"], 0))
    print("Open time slot:", time_map.get(row["id"], "N/A"))
    restaurant = build_restaurant(row, tag_map.get(row["id"], []), review_map.get(row["id"], 0), time_map.get(row["id"], "N/A"))
    restaurants.append(restaurant)
for r in restaurants:
    print(r)



Quán Dê nướng sa tế 0.5083712729110286
Categories: ['lá cách', 'bò nướng', 'rau cang cua', 'nấm', 'dê nướng', 'cốt lết chiên', 'món dê', 'món xào chay', 'cốt lết', 'goi bo rau cang cua', 'món bò dễ làm', 'món nướng dễ làm', 'thịt bò', 'gỏi', 'húng chanh', 'cải ngồng', 'cải xào']
Review count: 28
Open time slot: noon
Quán Mì nưa xào kim chi thịt bò 0.5042272391244589
Categories: ['món mì xào', 'món ngon từ hàu', 'mì xào', 'món cháo ngon', 'canh sườn', 'cam', 'nha đam', 'thịt kho', 'món kho', 'thịt bò', 'cháo', 'món cháo ngon dễ nấu', 'mì nưa', 'trứng cút kho', 'hàu', 'cách kho thịt ngon', 'món cháo', 'ba chỉ', 'kim chi', 'chả lụa', 'mì xào kim chi']
Review count: 32
Open time slot: all_day
Quán Gỏi Thịt Bò Cà Pháo 0.4915718054293585
Categories: ['gỏi thịt bò cà pháo', 'gỏi đu đủ tôm tai heo', 'sữa chua úp ngược', 'ốc hương rang muối sợi', 'thạch sữa bắp lá dứa']
Review count: 31
Open time slot: noon
Quán Bún thịt heo xiên sả nướng 0.4895171439912215
Categories: ['bún thịt nướng', 'nghệ 

In [48]:
#FEATURE ENGINEERING MODULE (FULL CODE)
import numpy as np
from collections import Counter
from datetime import datetime
TIME_SLOT_MAP = {
    "morning": [1, 0, 0, 0],
    "noon":    [0, 1, 0, 0],
    "afternoon":[0, 0, 1, 0],
    "evening": [0, 0, 0, 1],
    "all_day": [1, 1, 1, 1]
}
def get_time_context():
    hour = datetime.now().hour

    return (
        0 if 6 <= hour < 11 else     # morning
        1 if 11 <= hour < 14 else    # noon
        2 if 14 <= hour < 18 else    # afternoon
        3                            # evening
    )
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km

    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def time_match_feature(time_context, open_time_slot):
    slot_vector = TIME_SLOT_MAP.get(open_time_slot, [0,0,0,0])
    return slot_vector[time_context]
def safe_log(x):
    return np.log1p(max(x, 0))
def time_soft_match(time_context, open_time_slot):
    slot_vector = TIME_SLOT_MAP.get(open_time_slot, [0,0,0,0])

    # nếu không match exact → vẫn có điểm nhẹ
    return 1.0 if slot_vector[time_context] == 1 else 0.2


In [54]:
def build_features(user, restaurant, global_stats=None):
    
    # ===================== SEMANTIC =====================
    semantic_score = restaurant.get("semantic_score", 0.0)

    # ===================== LOCATION =====================
    distance_km = haversine(
        user["lat"], user["lon"],
        restaurant["lat"], restaurant["lng"]
    )

    distance_score = 1 / (1 + distance_km)

    nearby_bucket = (
        3 if distance_km <= 1 else
        2 if distance_km <= 3 else
        1 if distance_km <= 5 else 0
    )

    # ===================== PRICE =====================
    price_level = restaurant.get("price_level", 0)
    user_budget = user.get("budget_level", 0)

    price_match = 1 if price_level == user_budget else 0
    price_distance = abs(price_level - user_budget)

    # ===================== QUALITY =====================
    rating = restaurant.get("rating", 0.0)
    review_count = restaurant.get("review_count", 0)

    rating_score = rating / 5.0
    review_score = safe_log(review_count)

    # fallback global avg
    avg_rating = global_stats.get("avg_rating", 4.0) if global_stats else 4.0
    m = 50

    weighted_rating = (
        (review_count / (review_count + m)) * rating +
        (m / (review_count + m)) * avg_rating
    )

    # ===================== POPULARITY =====================
    popularity_score = safe_log(review_count)

  
    time_slot_vector = TIME_SLOT_MAP.get(restaurant.get("open_time_slot"), [0,0,0,0])
    time_context = get_time_context()
    time_match = time_match_feature(time_context, restaurant.get("open_time_slot"))
    time_soft = time_soft_match(time_context, restaurant.get("open_time_slot"))
    # ===================== TAG FEATURES =====================
   
    restaurant_tags = restaurant.get("categories", [])

    # query_tags = user.get("query_tags", [])

    # tag_overlap_score = tag_overlap(query_tags, restaurant_tags)
    # tag_jaccard_score = tag_jaccard(query_tags, restaurant_tags)
    tag_richness_score = safe_log(len(restaurant_tags))

    # ===================== FINAL VECTOR =====================
    features = [
        semantic_score,

        distance_km,
        distance_score,
        nearby_bucket,

        price_level,
        price_match,
        price_distance,

        rating_score,
        review_score,
        weighted_rating,

        popularity_score,

        time_match,
        time_soft,
        *time_slot_vector,
        time_context,

        # tag_overlap_score,
        # tag_jaccard_score,
        tag_richness_score
    ]

    return np.array(features, dtype=float)

In [56]:
# nếu muốn build feature cho từng quán
features = [build_features(user_context, r) for r in restaurants]
features = np.vstack(features)   # ma trận [num_restaurants, num_features]
print(features)

[[0.50837127 3.80940471 0.20792594 1.         3.         0.
  1.         0.54071429 3.36729583 3.53461538 3.36729583 1.
  1.         0.         1.         0.         0.         1.
  2.89037176]
 [0.50422724 4.30972958 0.18833351 1.         1.         0.
  1.         0.565      3.49650756 3.54146341 3.49650756 1.
  1.         1.         1.         1.         1.         1.
  3.09104245]
 [0.49157181 4.73032051 0.17451031 1.         2.         1.
  0.         0.61586022 3.4657359  3.64763374 3.4657359  1.
  1.         0.         1.         0.         0.         1.
  1.79175947]
 [0.48951714 2.67750526 0.27192347 2.         3.         0.
  1.         0.54729885 3.40119738 3.53618143 3.40119738 0.
  0.2        1.         0.         0.         0.         1.
  2.77258872]
 [0.47950268 3.71238445 0.21220679 1.         3.         0.
  1.         0.53709677 3.4657359  3.49691358 3.4657359  1.
  1.         0.         1.         0.         0.         1.
  2.7080502 ]]


In [58]:
import numpy as np

FEATURE_NAMES = [
"semantic_score",
"distance_km",
"distance_score",
"nearby_bucket",
"price_level",
"price_match",
"price_distance",
"rating_score",
"review_score",
"weighted_rating",
"popularity_score",
"time_match",
"time_soft",
"open_morning",
"open_noon",
"open_afternoon",
"open_evening",
"time_context",
"tag_richness_score"
]
def explain_feature_vector(vec, names=FEATURE_NAMES, decimals=6):
    v = np.asarray(vec).flatten()
    if len(v) != len(names):
        print(f"Canh bao: so feature = {len(v)} nhung so nhan = {len(names)}")
    n = min(len(v), len(names))
    for i in range(n):
        print(f"{i+1:02d}. {names[i]:20s}: {v[i]:.{decimals}f}")
# giải thích feature vector của quán đầu tiên

explain_feature_vector(features[0])

01. semantic_score      : 0.508371
02. distance_km         : 3.809405
03. distance_score      : 0.207926
04. nearby_bucket       : 1.000000
05. price_level         : 3.000000
06. price_match         : 0.000000
07. price_distance      : 1.000000
08. rating_score        : 0.540714
09. review_score        : 3.367296
10. weighted_rating     : 3.534615
11. popularity_score    : 3.367296
12. time_match          : 1.000000
13. time_soft           : 1.000000
14. open_morning        : 0.000000
15. open_noon           : 1.000000
16. open_afternoon      : 0.000000
17. open_evening        : 0.000000
18. time_context        : 1.000000
19. tag_richness_score  : 2.890372


In [ ]:
# Good restaurant nếu:
# - semantic cao
# - gần
# - rating tốt
# - mở đúng giờ
# - hợp giá
FEATURE_INDEX = {
    "semantic_score": 0,
    "distance_km": 1,
    "distance_score": 2,
    "nearby_bucket": 3,
    "price_level": 4,
    "price_match": 5,
    "price_distance": 6,
    "rating_score": 7,
    "review_score": 8,
    "weighted_rating": 9,
    "popularity_score": 10,
    "time_match": 11,
    "time_soft": 12,
    "open_morning": 13,
    "open_noon": 14,
    "open_afternoon": 15,
    "open_evening": 16,
    "time_context": 17,
    "tag_richness_score": 18
}


In [60]:
import numpy as np

def generate_pseudo_labels(X):
    y = []

    for row in X:
        score = 0

        # 🔥 CORE relevance
        score += 0.4 * row[FEATURE_INDEX["semantic_score"]]

        # 🔥 distance (prefer gần)
        score += 0.15 * row[FEATURE_INDEX["distance_score"]]

        # 🔥 quality
        score += 0.15 * row[FEATURE_INDEX["rating_score"]]
        score += 0.05 * row[FEATURE_INDEX["weighted_rating"]]

        # 🔥 popularity
        score += 0.05 * row[FEATURE_INDEX["popularity_score"]]

        # 🔥 price fit
        score += 0.05 * row[FEATURE_INDEX["price_match"]]
        score -= 0.05 * row[FEATURE_INDEX["price_distance"]]

        # 🔥 time context
        score += 0.05 * row[FEATURE_INDEX["time_match"]]
        score += 0.03 * row[FEATURE_INDEX["time_soft"]]

        # 🔥 diversity nhẹ
        score += 0.02 * row[FEATURE_INDEX["tag_richness_score"]]

        y.append(score)

    return np.array(y)

In [61]:
def convert_to_rank_labels(y, n_bins=5):
    """
    Convert continuous score → ranking levels (0 → 4)
    """
    bins = np.percentile(y, np.linspace(0, 100, n_bins + 1))
    labels = np.digitize(y, bins[1:-1])
    return labels

In [65]:
import lightgbm as lgb

# def train_ltr(X, y, group):
#     model = lgb.LGBMRanker(
#         objective="lambdarank",
#         metric="ndcg",
#         boosting_type="gbdt",
#         n_estimators=300,
#         learning_rate=0.05,
#         num_leaves=31,
#         max_depth=-1
#     )

#     model.fit(
#         X,
#         y,
#         group=group
#     )

#     return model
import lightgbm as lgb

def train_ltr(X, y, group):
    model = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        boosting_type="gbdt",
        n_estimators=30,
        learning_rate=0.05,
        num_leaves=7,
        max_depth=3,
        min_child_samples=1,   # quan trọng
        min_data_in_bin=1,     # quan trọng
        verbosity=-1
    )
    model.fit(X, y, group=group)
    return model

In [67]:
# X = feature matrix của bạn
X = features  # chính là cái bạn gửi

# STEP 1: pseudo label
y_continuous = generate_pseudo_labels(X)

# STEP 2: convert sang ranking label
y = convert_to_rank_labels(y_continuous)

# STEP 3: group
group = [len(X)]
print("X shape:", X.shape)
print("y unique:", np.unique(y))
print("Feature variance:", np.var(X, axis=0))
# STEP 4: train
model = train_ltr(X, y, group)
print(model.feature_importances_)

X shape: (5, 19)
y unique: [0 1 2 3 4]
Feature variance: [1.09051802e-04 4.76324541e-01 1.11356822e-03 1.60000000e-01
 6.40000000e-01 1.60000000e-01 1.60000000e-01 8.39210174e-04
 2.26136578e-03 2.56835199e-03 2.26136578e-03 1.60000000e-01
 1.02400000e-01 2.40000000e-01 1.60000000e-01 1.60000000e-01
 1.60000000e-01 0.00000000e+00 2.01453750e-01]
[78 37  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]


In [68]:
def rerank(model, X):
    scores = model.predict(X)
    ranked_idx = np.argsort(scores)[::-1]
    return ranked_idx, scores

In [69]:
ranked_idx, scores = rerank(model, X)

print("Ranking:", ranked_idx)
print("Scores:", scores)

Ranking: [2 1 0 4 3]
Scores: [-0.90772297  0.16777788  1.96927745 -2.0651246  -1.65335711]


d:\data science\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


In [70]:
for i, row in enumerate(X):
    print(i, row[0], row[2], row[7])  # semantic, distance_score, rating

0 0.5083712729110286 0.20792594112319357 0.540714285714286
1 0.5042272391244589 0.1883335084714255 0.5650000000000001
2 0.4915718054293585 0.17451030856906016 0.615860215053764
3 0.4895171439912215 0.27192347243451137 0.547298850574712
4 0.47950267702473526 0.21220679465090886 0.5370967741935481


In [71]:
for idx in ranked_idx:
    print(idx, scores[idx])

2 1.9692774522103433
1 0.16777787786802217
0 -0.9077229659362159
4 -1.6533571094397435
3 -2.065124597634516


In [17]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json

def get_best_dish_semantic(restaurant_ids, food, model):

    # ===== LOAD MENU + EMBEDDING =====
    query = """
    SELECT restaurant_id, dish_name, embedding
    FROM menus
    WHERE restaurant_id = ANY(%s)
    """
    menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))

    # convert embedding string → vector
    menu_df["embedding"] = menu_df["embedding"].apply(
        lambda x: np.array(json.loads(x)) if isinstance(x, str) else np.array(x)
    )

    # ===== EMBEDDING QUERY =====
    query_emb = model.encode(food)

    # ===== TÍNH SIMILARITY =====
    menu_df["score"] = menu_df["embedding"].apply(
        lambda emb: cosine_similarity([query_emb], [emb])[0][0]
    )

    # ===== LẤY BEST MỖI QUÁN =====
    best = (
        menu_df
        .sort_values("score", ascending=False)
        .groupby("restaurant_id")
        .first()
    )

    return best[["dish_name", "score"]].to_dict(orient="index")

In [18]:
food = parsed["slots"].get("food")

best_dishes = get_best_dish_semantic(
    tfidf_result["id"].tolist(),
    food,
    model
)
print("\n===== TF-IDF RESULT =====")

for _, row in tfidf_result.iterrows():

    rid = row["id"]

    if rid in best_dishes:
        dish = best_dishes[rid]["dish_name"]
    else:
        dish = "Không tìm thấy món phù hợp"

    print(f"\n🍜 {row['name']} (⭐ {row['rating']:.1f})")
    print(f"👉 Món phù hợp: {dish}")
best_dishes_emb = get_best_dish_semantic(
    embedding_result["id"].tolist(),
    food,
    model
)

print("\n===== EMBEDDING RESULT =====")

for _, row in embedding_result.iterrows():

    rid = row["id"]

    if rid in best_dishes_emb:
        dish = best_dishes_emb[rid]["dish_name"]
    else:
        dish = "Không tìm thấy món phù hợp"

    print(f"\n🍜 {row['name']} (⭐ {row['rating']:.1f})")
    print(f"👉 Món phù hợp: {dish}")

C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1067276177.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))



===== TF-IDF RESULT =====

🍜 Quán Bột chiên (⭐ 2.5)
👉 Món phù hợp: Miến xào chay

🍜 Quán Canh bí đỏ rong biển (⭐ 2.7)
👉 Món phù hợp: Mực xào nấm hương

🍜 Quán Thịt quay kho cà pháo (⭐ 2.5)
👉 Món phù hợp: Salad cua thanh

🍜 Quán Udon thịt xíu (⭐ 3.1)
👉 Món phù hợp: Bánh đa cua

🍜 Quán Lẩu ghẹ kim chi (⭐ 2.7)
👉 Món phù hợp: Phở xào chua ngọt

===== EMBEDDING RESULT =====

🍜 Quán Bê thui xào sa tế (⭐ 2.8)
👉 Món phù hợp: Canh bó xôi bò viên

🍜 Quán Chả Hoa Ngũ Sắc Phô Mai (⭐ 2.9)
👉 Món phù hợp: Chả Hoa Ngũ Sắc Phô Mai

🍜 Quán Gà kho rau củ (⭐ 2.8)
👉 Món phù hợp: Gà kho rau củ

🍜 Quán Salad cá hồi (⭐ 2.7)
👉 Món phù hợp: Salad cá hồi

🍜 Quán Miến xào nghêu tay cầm (⭐ 2.6)
👉 Món phù hợp: Miến xào nghêu tay cầm


C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1067276177.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))


In [8]:
def get_menus(restaurant_ids):

    query = f"""
    SELECT restaurant_id, dish_name
    FROM menus
    WHERE restaurant_id = ANY(%s)
    """

    menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))

    # group thành list
    menu_grouped = menu_df.groupby("restaurant_id")["dish_name"].apply(list).to_dict()

    return menu_grouped

In [9]:
tfidf_result = search_tfidf(query)

menu_map = get_menus(tfidf_result["id"].tolist())

tfidf_result["menu"] = tfidf_result["id"].map(menu_map)

C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1255267676.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))


In [10]:
embedding_result = search_embedding(query)

menu_map2 = get_menus(embedding_result["id"].tolist())

embedding_result["menu"] = embedding_result["id"].map(menu_map2)

C:\Users\Admin\AppData\Local\Temp\ipykernel_24336\1255267676.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  menu_df = pd.read_sql(query, conn, params=(restaurant_ids,))


In [11]:
print("\n===== TF-IDF RESULT =====")
for _, row in tfidf_result.iterrows():
    print(f"\n🍜 {row['name']} (⭐ {row['rating']:.1f})")
    print("Menu:", ", ".join(row["menu"][:5]))

print("\n===== EMBEDDING RESULT =====")
for _, row in embedding_result.iterrows():
    print(f"\n🍜 {row['name']} (⭐ {row['rating']:.1f})")
    print("Menu:", ", ".join(row["menu"][:5]))


===== TF-IDF RESULT =====

🍜 Quán Bánh Tráng Trộn Ngon (⭐ 2.9)
Menu: Bánh Tráng Trộn Ngon, Chả Cá Lã Vọng, Khoai Môn Kẹp Thịt Chiên Giòn, Giả Khoai Tây Chiên Từ Bánh, Mứt Mận

🍜 Quán Rau Câu Bánh Trung Thu (⭐ 2.6)
Menu: Bò Nấu Bia, Bánh Cuốn Tôm, Bò Chiên Kiểu Thái, Rau Câu Bánh Trung Thu, Soup Gà

🍜 Quán Thịt Heo Giả Khô Bò (⭐ 2.9)
Menu: Thịt Heo Giả Khô Bò, Bò Kho Gừng, Xoài Lắc, Gà Sốt Chua Ngọt Tẩm Mè, Chả Ram Chiên Giòn

🍜 Quán Bánh Kẹp Tàn Ong (⭐ 2.6)
Menu: Thịt Ba Rọi Chiên Giòn, Hủ Tiếu Khô, Bánh Kẹp Tàn Ong, Bạch Tuộc Xào Cần Tỏi, Phở Cuốn Thịt Bò Rau Thơm

🍜 Quán Bún Bò Nam Bộ (⭐ 2.5)
Menu: Bún Bò Nam Bộ, Thịt Bò Xào Dưa Cải Chua, Bò Bít Tết Dầu Hào Maggi, Milo Đá Bào, Làm Bánh Bò Nướng

===== EMBEDDING RESULT =====

🍜 Quán Thịt Bò Kho Sả (⭐ 2.8)
Menu: Mì Tươi Trộn Sốt Bò Bằm, Rượu Trái Cây, Lòng Heo Nướng, Mực Dồn Thịt Rim Kẹo, Thịt Bò Kho Sả

🍜 Quán Thịt Bò Viên Thơm Ngon (⭐ 3.2)
Menu: Thịt Bò Viên Thơm Ngon, Bánh Chuối Bí Ngô Chiên Giòn, Bì Cuốn, Bánh Quy Bơ Hình Hoa Cúc,